# Team composition under two opportunity rules

Leo and Charles • first LG exploration demonstration • 6 October 2026

We compare **LG remaining-seat opportunities** with a separately labeled
**one-opportunity-per-open-team alternative**. Homophily is the only preference
in either rule. All teams finish with exactly 20 players, so composition changes
are examined at identical final roster sizes. No SCORE, SELECT, or MLE is run.

The reviewed demonstration uses 2,000 N(0,1) players, 100 teams, five rho values,
and 30 paired assignment repetitions (300 divisions). Charles controls execution.
The notebook has not been run on the approved full settings yet.


## Settings and execution control

Start Jupyter from this exploration folder (or its notebooks/ directory), and
choose a kernel with NumPy and Matplotlib. The existing sports_net environment
is suitable. Scientific code uses relative project paths and no legacy imports.

Review the visible settings below. Set RUN_EXPERIMENT=True to run or resume the
approved demonstration, then run the notebook. Changing settings creates a new
scientific run ID. Leave the switch False to view previously completed results.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'README_START_HERE.md').is_file():
    raise RuntimeError('Start Jupyter in lg_exploration/ or lg_exploration/notebooks/.')

sys.path.insert(0, str(PROJECT_ROOT / 'code'))
from lg_composition.experiment import (
    Settings, run_experiment, run_directory, load_complete, plot_results,
)

# Execution belongs to Charles. False prevents an accidental simulation run.
RUN_EXPERIMENT = False

# These are the values agreed in our discussion. Edit explicitly when planning
# a later experiment; the saved run settings preserve every executed choice.
SETTINGS = {
    'n_players': 2000,
    'n_teams': 100,
    'roster_size': 20,
    'rhos': [0.0, 0.5, 1.0, 2.0, 4.0],
    'assignment_repetitions': 30,
    'population_seed': 6102026,
    'assignment_seed': 6102027,
    'results_directory': 'results/composition_first',
}
settings = Settings.from_dict(SETTINGS)
print(json.dumps(SETTINGS, indent=2))
print(f'Planned divisions: {settings.unit_count}')
print(f'Checkpoint and results directory: {run_directory(settings, PROJECT_ROOT)}')


## What the two rules do

For incoming player i, team j has centroid T_j and remaining seats R_j.

- LG seat opportunities: weight = R_j × exp(-rho × abs(A_i - T_j)).
- Team opportunities: weight = exp(-rho × abs(A_i - T_j)) for open teams.

Both exclude full teams. Every empty centroid starts at the realized population
mean; its first member replaces that signal. Normal draws retain their original
sample moments. The numerical implementation shifts log weights before
normalizing, preserving the equations even when direct exponentials underflow.

Population, arrival-order and choice streams are separate. Each repetition
shares its arrival order and underlying inverse-CDF uniform draws across all
rho values and both rules. Their assignments can diverge despite those shared
inputs. Stable seeds are independent of task scheduling.


In [ ]:
if RUN_EXPERIMENT:
    completed_directory = run_experiment(settings, PROJECT_ROOT)
    print(f'Complete: {completed_directory}')
else:
    print('Execution switch is False. No divisions computed in this cell.')


## Verify completed divisions before plotting

Every checkpoint includes player identities, original talent, assignment,
capacities, final centroids, variance, settings identifiers, seeds and random
inputs. A completed unit is atomically saved and validated before reuse.
Interrupting the notebook preserves completed units; rerun to resume. Damaged
checkpoints produce an explicit error and are preserved rather than silently
accepted or overwritten. Do not run two notebook/CLI processes on the same run.


In [ ]:
RESULTS_READY = False
completed_directory = run_directory(settings, PROJECT_ROOT)
if (completed_directory / 'run_metadata.json').exists():
    metadata = json.loads((completed_directory / 'run_metadata.json').read_text())
    if metadata.get('status') == 'complete':
        completed_directory, divisions = load_complete(settings, PROJECT_ROOT)
        RESULTS_READY = True
        print(f'Verified {len(divisions)} divisions.')
        print(f'Realized population mean: {metadata["population_mean"]:.5f}')
        print(f'Realized population SD: {metadata["population_sd"]:.5f}')
        print(f'Random-grouping expected H_sort: {metadata["expected_random_h"]:.5f}')
    else:
        print('Run is incomplete; set RUN_EXPERIMENT=True to resume before plotting.')
else:
    print('The approved demonstration has not been executed. Set the switch True to begin.')


## Read the figures

Five figures are saved as PNG and editable SVG:

1. **Team means:** five consistently colored curves per rule. At x, the curve
   gives the fraction of teams with mean talent at or below x. This is an
   empirical cumulative distribution, averaged over assignments.
2. **Team variances:** the same display for within-team population variance
   (ddof=0). A leftward shift means more similar talent within teams.
3. **Division summaries:** sorting, centroid spread, and mean within-team
   variance across rho. Bars are ±1 assignment SD, conditional on this population.
4. **Paired sorting differences:** team-rule minus seat-rule H_sort within each
   repetition, averaged across repetitions. Bars show the SD of paired differences.
5. **Actual rosters:** a heatmap of the first paired repetition, selected in
   advance. Teams are independently ranked by their final mean in every panel;
   rows are ranks, not matching team identities. Members are sorted within teams.

Shading is ±1 SD across division-level distribution curves, not a confidence
interval based on pooling teams. The five rho colors are identical in both
CDF figures. Scalar summaries use color/line style for the two opportunity rules.


In [ ]:
if RESULTS_READY:
    import matplotlib.pyplot as plt
    figures, plot_directory = plot_results(settings, PROJECT_ROOT)
    plt.show()
    print(f'PNG and SVG figures saved in {plot_directory}')


## Scientific checks and limits

At rho=0, both rules have the same expected H_sort, (J-1)/(M-1), with random
arrival and fully filled equal-size teams. Neither must produce H_sort=0.
Individual paired assignments may differ, and finite-repeat averages may differ.

With equal final sizes and population-variance conventions,

population talent variance = variance of team means + mean within-team variance.

Consequently H_sort equals variance of team means divided by population talent
variance. These summaries explain the same decomposition; they are not three
independent pieces of evidence. The distribution plots and roster heatmap add
composition detail beyond that decomposition.

Thirty assignments measure assignment variability for **one** population.
They do not establish robustness across fresh talent populations, nor determine
the direction of the rule difference in advance. No desired curve is assumed.

Excess capacity and convergence toward the uncapped rule are later experiments.
Their cap grids have not been chosen. This first test does not demonstrate
large-cap convergence because it deliberately fills all prescribed seats.

See documents/LEO_DECISION_LOG.md and documents/DEPENDENCY_LOG.md for decisions
and provenance. Source snapshots remain untouched. Bulk output belongs in rsync,
not Git; synchronization must be configured before larger runs.
